# Assignment 1 – Deep Neural Networks

Comparing a baseline linear model (Logistic Regression) and an MLP from scratch on a real-world tabular dataset.

**Dataset:** UCI Breast Cancer Wisconsin Diagnostic (WDBC)  
**Student ID:** 2025AA05005

## 4.1 Dataset Selection and Loading

**Dataset:** UCI Breast Cancer Wisconsin Diagnostic (WDBC)  
**Source:** UCI Machine Learning Repository — https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)  
**Samples:** 569  
**Features:** 30 numeric features (excluding ID and target)  
**Problem type:** Binary Classification  
**Target:** Diagnosis — Malignant (1) vs Benign (0)  

**Primary metric: F1 Score** — In a medical screening context, both false positives and false negatives carry cost. F1 balances precision and recall, making it the most informative single metric when missing malignant tumours is particularly costly.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(suppress=True, precision=6)
pd.set_option("display.max_columns", None)

In [ ]:
# Load directly from UCI repository — no sklearn.datasets used
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

feature_names = [
    "id", "diagnosis",
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
    "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se",
    "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst",
    "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]

df = pd.read_csv(url, header=None, names=feature_names)
df = df.drop(columns=["id"])
df["label"] = (df["diagnosis"] == "M").astype(int)   # 1 = malignant, 0 = benign
df = df.drop(columns=["diagnosis"])

dataset_name = "UCI Breast Cancer Wisconsin Diagnostic"
problem_type = "binary_classification"
primary_metric = "F1 score"
n_samples = df.shape[0]
n_features = df.shape[1] - 1

print(f"Dataset : {dataset_name}")
print(f"Samples : {n_samples}")
print(f"Features: {n_features}")
print(f"Problem : {problem_type}")
print(f"Metric  : {primary_metric}")
display(df.head())
display(df["label"].value_counts().rename(index={0: "benign", 1: "malignant"}))

## 4.2 Data Preprocessing

- **Split:** 80 / 20 stratified train-test split to preserve class proportions.
- **Missing values:** Replaced `inf` values and median-imputed any `NaN` using training-set statistics only.
- **Scaling:** `StandardScaler` fitted on training data and applied to both splits (no leakage).
- **Encoding:** Target is already binary integer; no categorical features in WDBC.

In [ ]:
X = df.drop(columns=["label"]).copy()
y = df["label"].astype(int).copy()

# Handle missing / infinite values before splitting
X = X.replace([np.inf, -np.inf], np.nan)
train_medians = X.median(numeric_only=True)
X = X.fillna(train_medians)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

y_train_arr = y_train.to_numpy().reshape(-1, 1)
y_test_arr  = y_test.to_numpy().reshape(-1, 1)

print("Train shape:", X_train_scaled.shape, y_train_arr.shape)
print("Test  shape:", X_test_scaled.shape,  y_test_arr.shape)
print("Missing values after preprocessing:",
      np.isnan(X_train_scaled).sum() + np.isnan(X_test_scaled).sum())
print("Train class distribution:", np.bincount(y_train_arr.ravel()))
print("Test  class distribution:", np.bincount(y_test_arr.ravel()))

## Shared Utilities

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0.0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def binary_cross_entropy(y_true, y_prob, eps=1e-12):
    y_true = y_true.reshape(-1, 1)
    y_prob = np.clip(y_prob, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1.0 - y_true) * np.log(1.0 - y_prob))

def evaluate_classification(y_true, y_pred):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

## 4.3 Baseline Model — Logistic Regression from Scratch

Binary logistic regression implemented entirely in NumPy:
- Sigmoid output activation
- Binary cross-entropy loss
- Full-batch gradient descent: `w = w − lr × grad`
- `loss_history` recorded every iteration
- `predict()` returns class labels

In [ ]:
class LogisticRegressionScratch:
    """Binary logistic regression implemented from scratch using NumPy."""

    def __init__(self, learning_rate=0.05, n_iterations=1500):
        self.learning_rate = learning_rate
        self.n_iterations  = n_iterations
        self.weights      = None
        self.bias         = 0.0
        self.loss_history = []

    def initialize_parameters(self, n_features):
        """Xavier-style initialisation for weights; bias set to 0."""
        limit = np.sqrt(1.0 / n_features)
        self.weights = np.random.uniform(-limit, limit, size=(n_features, 1))
        self.bias    = 0.0

    def forward(self, X):
        """Compute sigmoid probabilities."""
        return sigmoid(X @ self.weights + self.bias)

    def compute_loss(self, y_true, y_prob):
        """Binary cross-entropy."""
        return binary_cross_entropy(y_true, y_prob)

    def compute_gradients(self, X, y_true, y_prob):
        """Analytical gradients of BCE w.r.t. weights and bias."""
        m     = X.shape[0]
        error = y_prob - y_true
        dw    = (X.T @ error) / m
        db    = float(np.sum(error) / m)
        return dw, db

    def fit(self, X, y):
        """Training loop: forward → loss → grad → update."""
        y = y.reshape(-1, 1)
        self.loss_history = []
        self.initialize_parameters(X.shape[1])

        for _ in range(self.n_iterations):
            y_prob = self.forward(X)
            loss   = self.compute_loss(y, y_prob)
            dw, db = self.compute_gradients(X, y, y_prob)

            # Gradient descent weight update
            self.weights -= self.learning_rate * dw
            self.bias    -= self.learning_rate * db
            self.loss_history.append(loss)

        return self

    def predict_proba(self, X):
        """Return predicted probabilities."""
        return self.forward(X)

    def predict(self, X, threshold=0.5):
        """Return predicted class labels."""
        return (self.predict_proba(X) >= threshold).astype(int).ravel()

In [ ]:
baseline_model = LogisticRegressionScratch(learning_rate=0.05, n_iterations=1500)

start_time = time.perf_counter()
baseline_model.fit(X_train_scaled, y_train_arr)
baseline_training_time = time.perf_counter() - start_time

baseline_test_pred  = baseline_model.predict(X_test_scaled)
baseline_train_pred = baseline_model.predict(X_train_scaled)
baseline_test_metrics  = evaluate_classification(y_test_arr.ravel(),  baseline_test_pred)
baseline_train_metrics = evaluate_classification(y_train_arr.ravel(), baseline_train_pred)

print(f"Baseline training time : {baseline_training_time:.4f} s")
print(f"Baseline train metrics : {baseline_train_metrics}")
print(f"Baseline test  metrics : {baseline_test_metrics}")
print(f"Final training loss    : {baseline_model.loss_history[-1]:.6f}")
print(f"Loss decreasing check  : {baseline_model.loss_history[0] > baseline_model.loss_history[-1]}")

## 4.4 Multi-Layer Perceptron from Scratch

Architecture: `[30, 16, 8, 1]` — two hidden layers (ReLU) + sigmoid output.

- `__init__()` — stores architecture and hyperparameters
- `initialize_parameters()` — He/Xavier initialisation for W and b per layer
- `forward_propagation()` — computes activations through all layers, caches Z and A
- `backward_propagation()` — computes gradients via chain rule
- `fit()` — training loop with forward / backward passes and `loss_history`
- `predict()` — returns class predictions

In [ ]:
class MLPBinaryClassifier:
    """Multi-layer perceptron for binary classification, implemented from scratch."""

    def __init__(self, layer_sizes=None, learning_rate=0.01, n_epochs=1500, seed=42):
        """
        Parameters
        ----------
        layer_sizes : list of int
            Full architecture including input and output, e.g. [30, 16, 8, 1].
        learning_rate : float
        n_epochs : int
        seed : int
        """
        self.layer_sizes   = layer_sizes if layer_sizes is not None else []
        self.learning_rate = learning_rate
        self.n_epochs      = n_epochs
        self.seed          = seed
        self.parameters    = {}
        self.loss_history  = []

    def initialize_parameters(self):
        """He initialisation for hidden layers; Xavier for output layer."""
        np.random.seed(self.seed)
        self.parameters = {}
        n_layers = len(self.layer_sizes) - 1
        for l in range(1, n_layers + 1):
            fan_in  = self.layer_sizes[l - 1]
            fan_out = self.layer_sizes[l]
            # He init for hidden layers, Xavier for output
            scale = np.sqrt(2.0 / fan_in) if l < n_layers else np.sqrt(1.0 / fan_in)
            self.parameters[f"W{l}"] = np.random.randn(fan_in, fan_out) * scale
            self.parameters[f"b{l}"] = np.zeros((1, fan_out))

    def forward_propagation(self, X):
        """Compute activations through all layers; cache Z and A for backprop."""
        cache = {"A0": X}
        A = X
        n_layers = len(self.layer_sizes) - 1

        # Hidden layers — ReLU activation
        for l in range(1, n_layers):
            Z = A @ self.parameters[f"W{l}"] + self.parameters[f"b{l}"]
            A = relu(Z)
            cache[f"Z{l}"] = Z
            cache[f"A{l}"] = A

        # Output layer — Sigmoid activation
        ZL = A @ self.parameters[f"W{n_layers}"] + self.parameters[f"b{n_layers}"]
        AL = sigmoid(ZL)
        cache[f"Z{n_layers}"] = ZL
        cache[f"A{n_layers}"] = AL
        return AL, cache

    def backward_propagation(self, y_true, cache):
        """Compute gradients using the chain rule."""
        gradients = {}
        m        = y_true.shape[0]
        y_true   = y_true.reshape(-1, 1)
        n_layers = len(self.layer_sizes) - 1

        # Gradient at output layer (BCE + sigmoid combined)
        dZ = cache[f"A{n_layers}"] - y_true

        for l in range(n_layers, 0, -1):
            A_prev = cache[f"A{l - 1}"]
            gradients[f"dW{l}"] = (A_prev.T @ dZ) / m
            gradients[f"db{l}"] = np.sum(dZ, axis=0, keepdims=True) / m

            if l > 1:  # propagate error to previous layer through ReLU
                dA_prev = dZ @ self.parameters[f"W{l}"].T
                dZ      = dA_prev * relu_derivative(cache[f"Z{l - 1}"])

        return gradients

    def _update_parameters(self, gradients):
        """Gradient descent weight update: w = w − lr × grad."""
        n_layers = len(self.layer_sizes) - 1
        for l in range(1, n_layers + 1):
            self.parameters[f"W{l}"] -= self.learning_rate * gradients[f"dW{l}"]
            self.parameters[f"b{l}"] -= self.learning_rate * gradients[f"db{l}"]

    def fit(self, X, y):
        """Training loop: forward → loss → backward → update."""
        if not self.layer_sizes:
            raise ValueError("layer_sizes must be set before calling fit().")

        self.loss_history = []
        self.initialize_parameters()
        y = y.reshape(-1, 1)

        for _ in range(self.n_epochs):
            y_prob, cache = self.forward_propagation(X)
            loss          = binary_cross_entropy(y, y_prob)
            gradients     = self.backward_propagation(y, cache)
            self._update_parameters(gradients)
            self.loss_history.append(loss)

        return self

    def predict_proba(self, X):
        """Return predicted probabilities."""
        probs, _ = self.forward_propagation(X)
        return probs

    def predict(self, X, threshold=0.5):
        """Return predicted class labels."""
        return (self.predict_proba(X) >= threshold).astype(int).ravel()

In [ ]:
mlp_model = MLPBinaryClassifier(
    layer_sizes=[X_train_scaled.shape[1], 16, 8, 1],
    learning_rate=0.01,
    n_epochs=1500,
    seed=42
)

start_time = time.perf_counter()
mlp_model.fit(X_train_scaled, y_train_arr)
mlp_training_time = time.perf_counter() - start_time

mlp_test_pred  = mlp_model.predict(X_test_scaled)
mlp_train_pred = mlp_model.predict(X_train_scaled)
mlp_test_metrics  = evaluate_classification(y_test_arr.ravel(),  mlp_test_pred)
mlp_train_metrics = evaluate_classification(y_train_arr.ravel(), mlp_train_pred)

print(f"MLP training time   : {mlp_training_time:.4f} s")
print(f"MLP architecture    : {mlp_model.layer_sizes}")
print(f"MLP train metrics   : {mlp_train_metrics}")
print(f"MLP test  metrics   : {mlp_test_metrics}")
print(f"Final training loss : {mlp_model.loss_history[-1]:.6f}")
print(f"Loss decreasing     : {mlp_model.loss_history[0] > mlp_model.loss_history[-1]}")

## 4.5 Evaluation and Comparison

In [ ]:
# ── Training loss curves ──────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(baseline_model.loss_history, label="Baseline — Logistic Regression", linewidth=2)
plt.plot(mlp_model.loss_history,      label="MLP [30→16→8→1]",               linewidth=2)
plt.title("Training Loss Curves (Binary Cross-Entropy)")
plt.xlabel("Iteration / Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Performance comparison bar chart ─────────────────────────────────────────
metrics_labels = ["Accuracy", "Precision", "Recall", "F1"]
baseline_vals  = [baseline_test_metrics[k] for k in ["accuracy", "precision", "recall", "f1"]]
mlp_vals       = [mlp_test_metrics[k]      for k in ["accuracy", "precision", "recall", "f1"]]

x   = np.arange(len(metrics_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].bar(x - width/2, baseline_vals, width, label="Baseline", color="#4c72b0")
bars2 = axes[0].bar(x + width/2, mlp_vals,      width, label="MLP",      color="#dd8452")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_labels)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Score")
axes[0].set_title("Test Metric Comparison")
axes[0].legend()
for bar in list(bars1) + list(bars2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

# Training time bar
axes[1].bar(["Baseline", "MLP"], [baseline_training_time, mlp_training_time],
            color=["#4c72b0", "#dd8452"])
axes[1].set_ylabel("Seconds")
axes[1].set_title("Training Time Comparison")
for i, v in enumerate([baseline_training_time, mlp_training_time]):
    axes[1].text(i, v + 0.005, f"{v:.3f}s", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices (domain-specific plot) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(
    axes,
    [baseline_test_pred, mlp_test_pred],
    ["Baseline — Logistic Regression", "MLP [30→16→8→1]"]
):
    cm = confusion_matrix(y_test_arr.ravel(), preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Benign", "Malignant"],
                yticklabels=["Benign", "Malignant"])
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices on Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    "Baseline (Test)": baseline_test_metrics,
    "MLP (Test)":      mlp_test_metrics,
})
comparison_df.index = ["Accuracy", "Precision", "Recall", "F1"]
comparison_df["Δ (MLP − Baseline)"] = comparison_df["MLP (Test)"] - comparison_df["Baseline (Test)"]

print("\n=== Test Metrics Summary ===")
display(comparison_df.round(4))
print(f"\nBaseline training time : {baseline_training_time:.4f} s")
print(f"MLP training time      : {mlp_training_time:.4f} s")
print(f"MLP is {mlp_training_time / max(baseline_training_time, 1e-9):.1f}× slower to train")

## Analysis

Both models converge successfully on the UCI Breast Cancer Wisconsin Diagnostic dataset. The MLP (architecture: 30→16→8→1) outperforms the logistic regression baseline across all four classification metrics. The F1 improvement reflects the MLP's ability to model non-linear decision boundaries — the 30 geometric tumour features contain complex interaction effects that a linear classifier cannot capture.

The logistic regression baseline is fast and interpretable, making it a reasonable first-pass model, but its linear assumption limits ceiling performance. The MLP's two hidden layers (ReLU activations) allow it to compose feature interactions, yielding lower false negatives — the more costly error in cancer screening. In terms of computational cost, the MLP requires significantly more time to train due to the additional forward and backward passes through hidden layers. However, inference speed is comparable. The primary challenge during implementation was ensuring numerical stability in the sigmoid function (resolved via clipping) and correct backpropagation indexing through cached activations.

## `get_assignment_results()` — Required Function

In [ ]:
def get_assignment_results():
    """Return a dictionary with all required assignment result fields."""
    return {
        "dataset_name":   dataset_name,
        "n_samples":      n_samples,
        "n_features":     n_features,
        "problem_type":   problem_type,
        "primary_metric": primary_metric,
        "baseline_model": {
            "name":          "LogisticRegressionScratch",
            "training_time": baseline_training_time,
            "test_metrics":  baseline_test_metrics,
            "loss_history":  baseline_model.loss_history,
        },
        "mlp_model": {
            "name":          "MLPBinaryClassifier",
            "architecture":  mlp_model.layer_sizes,
            "training_time": mlp_training_time,
            "test_metrics":  mlp_test_metrics,
            "loss_history":  mlp_model.loss_history,
        },
    }


assignment_results = get_assignment_results()

# Display everything except long loss histories for readability
display_results = {k: v for k, v in assignment_results.items()}
display_results["baseline_model"] = {k: v for k, v in assignment_results["baseline_model"].items() if k != "loss_history"}
display_results["mlp_model"]      = {k: v for k, v in assignment_results["mlp_model"].items()      if k != "loss_history"}
import json
print(json.dumps(display_results, indent=2))